In [2]:
import ee
import geemap
import pandas as pd
import numpy as np

In [4]:
# ee.Authenticate()
ee.Initialize(project='dev-airlock-454813-h7')

In [3]:
Map = geemap.Map(basemap='ROADMAP')
region = ee.FeatureCollection('FAO/GAUL/2015/level2') \
    .filter(ee.Filter.eq("ADM2_NAME","Kullu"))
style = {'color':'black','fillColor':'00000000','width':2}
Map.addLayer(region.style(**style), {}, 'Region')
Map.centerObject(region, 8)
Map

Map(center=[31.90736035698947, 77.38492629598922], controls=(WidgetControl(options=['position', 'transparent_b…

In [4]:
def add_s2_indices(image):
    image1 = image.select(['B8','B4','B3','B2','B5','B6','B7','B11','B12'])
    nir = image1.select('B8')
    red = image1.select('B4')
    green = image1.select('B3')
    blue = image1.select('B2')
    redge1 = image1.select('B5')
    redge2 = image1.select('B6')
    redge3 = image1.select('B7')
    swir1 = image1.select('B11')
    swir2 = image1.select('B12')

    ndvi = image1.normalizedDifference(['B8','B4']).rename('NDVI')

    evi = image1.expression(
        '2.5*((NIR-RED)/(NIR+6*RED-7.5*BLUE+1))',
        {'NIR':nir,'RED':red,'BLUE':blue}).rename('EVI')

    ipvi = image1.expression(
        '0.5*((NIR-RED)/(NIR+RED)+1)',
        {'NIR':nir,'RED':red}).rename('IPVI')

    tndvi = image1.expression(
        'sqrt((NIR-RED)/(NIR+RED)+0.5)',
        {'NIR':nir,'RED':red}).rename('TNDVI')

    gndvi = image1.normalizedDifference(['B8','B3']).rename('GNDVI')

    bi2 = image1.expression(
        'sqrt((RED**2+GREEN**2+NIR**2)/3)',
        {'RED':red,'GREEN':green,'NIR':nir}).rename('BI2')

    mtci = image1.expression(
        '(RE2-RE1)/(RE1-RED)',
        {'RE2':redge2,'RE1':redge1,'RED':red}).rename('MTCI')

    reip = image1.expression(
        '705 + 35*((RED+RE3)/2 - RE1)/(RE2-RE1)',
        {'RED':red,'RE3':redge3,'RE1':redge1,'RE2':redge2}).rename('REIP')

    ireci = image1.expression(
        '(RE3-RED)/(RE1/RE2)',
        {'RE3':redge3,'RED':red,'RE1':redge1,'RE2':redge2}).rename('IRECI')

    glcm = nir.toInt().glcmTexture(size=1) \
        .select(['B8_asm','B8_contrast','B8_idm','B8_ent'])

    return image1.addBands([
        swir1,swir2,ndvi,evi,ipvi,tndvi,gndvi,
        bi2,mtci,reip,ireci,glcm
    ])

In [5]:
def add_s1_indices(image):
    image1 = image.select(['VV','VH'])
    vv = image1.select('VV')
    vh = image1.select('VH')

    div = vh.divide(vv).rename('div')
    diff = vh.subtract(vv).rename('diff')

    amp = image1.expression(
        'sqrt(VH**2 + VV**2)',
        {'VH':vh,'VV':vv}).rename('amp')

    norm = vh.subtract(vv).divide(vh.add(vv)).rename('norm')

    return image1.addBands([div,diff,amp,norm])

In [6]:
season_months = [3, 6, 9, 12]
year = 2025

In [7]:
def get_seasonal_stack(month):

    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, 'month')

    s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
        .filterBounds(region) \
        .filterDate(start, end) \
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',30)) \
        .median().clip(region)
    s2 = add_s2_indices(s2)
    

    s1 = ee.ImageCollection("COPERNICUS/S1_GRD") \
        .filterBounds(region) \
        .filterDate(start, end) \
        .filter(ee.Filter.eq('instrumentMode','IW')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation','VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation','VH')) \
        .median().clip(region)
    s1 = add_s1_indices(s1)
    season_stack = s2.addBands(s1)
    return season_stack

In [8]:
season_images = []

for m in season_months:
    season_img = get_seasonal_stack(m)
    season_img = season_img.rename(
        season_img.bandNames().map(lambda b: ee.String(b).cat('_').cat(ee.Number(m).format()))
    )
    season_images.append(season_img)

srtm = ee.Image('USGS/SRTMGL1_003')
slope = ee.Terrain.slope(srtm).rename('Slope')
aspect = ee.Terrain.aspect(srtm).rename('Aspect')
terrain = slope.addBands(aspect)
stack = ee.Image.cat(season_images).addBands(terrain)
stack = stack.unmask(-9999)

In [9]:
print(stack.bandNames().getInfo())

['B8_3', 'B4_3', 'B3_3', 'B2_3', 'B5_3', 'B6_3', 'B7_3', 'B11_3', 'B12_3', 'B11_1_3', 'B12_1_3', 'NDVI_3', 'EVI_3', 'IPVI_3', 'TNDVI_3', 'GNDVI_3', 'BI2_3', 'MTCI_3', 'REIP_3', 'IRECI_3', 'B8_asm_3', 'B8_contrast_3', 'B8_idm_3', 'B8_ent_3', 'VV_3', 'VH_3', 'div_3', 'diff_3', 'amp_3', 'norm_3', 'B8_6', 'B4_6', 'B3_6', 'B2_6', 'B5_6', 'B6_6', 'B7_6', 'B11_6', 'B12_6', 'B11_1_6', 'B12_1_6', 'NDVI_6', 'EVI_6', 'IPVI_6', 'TNDVI_6', 'GNDVI_6', 'BI2_6', 'MTCI_6', 'REIP_6', 'IRECI_6', 'B8_asm_6', 'B8_contrast_6', 'B8_idm_6', 'B8_ent_6', 'VV_6', 'VH_6', 'div_6', 'diff_6', 'amp_6', 'norm_6', 'B8_9', 'B4_9', 'B3_9', 'B2_9', 'B5_9', 'B6_9', 'B7_9', 'B11_9', 'B12_9', 'B11_1_9', 'B12_1_9', 'NDVI_9', 'EVI_9', 'IPVI_9', 'TNDVI_9', 'GNDVI_9', 'BI2_9', 'MTCI_9', 'REIP_9', 'IRECI_9', 'B8_asm_9', 'B8_contrast_9', 'B8_idm_9', 'B8_ent_9', 'VV_9', 'VH_9', 'div_9', 'diff_9', 'amp_9', 'norm_9', 'B8_12', 'B4_12', 'B3_12', 'B2_12', 'B5_12', 'B6_12', 'B7_12', 'B11_12', 'B12_12', 'B11_1_12', 'B12_1_12', 'NDVI_12',

In [10]:
df = pd.read_csv("istp.csv")

features = []

for _, row in df.iterrows():

    lon = float(row['Longitude'])
    lat = float(row['Latitude'])

    f = ee.Feature(
        ee.Geometry.Point([lon, lat]),
        {'class': int(row['no'])}
    )

    features.append(f)

training_fc = ee.FeatureCollection(features)

Map.addLayer(training_fc, {'color':'red'}, 'Training Points')

print("Training points:", training_fc.size().getInfo())

Training points: 72


In [11]:
Map.addLayer(training_fc, {'color':'red'}, 'Training Points')
Map

Map(bottom=26857.0, center=[32.263910555201306, 77.3822021484375], controls=(WidgetControl(options=['position'…

In [12]:
bands = stack.bandNames()
bands = ['B8_3', 'B4_3', 'B3_3', 'B2_3', 'B5_3', 'B6_3', 'B7_3', 'B11_3', 'B12_3', 'B11_1_3', 'B12_1_3', 'NDVI_3', 'EVI_3', 'IPVI_3', 'TNDVI_3', 'GNDVI_3', 'BI2_3', 'MTCI_3', 'REIP_3', 'IRECI_3', 'VV_3', 'VH_3', 'B8_6', 'B4_6', 'B3_6', 'B2_6', 'B5_6', 'B6_6', 'B7_6', 'B11_6', 'B12_6', 'B11_1_6', 'B12_1_6', 'NDVI_6', 'EVI_6', 'IPVI_6', 'TNDVI_6', 'GNDVI_6', 'BI2_6', 'MTCI_6', 'REIP_6', 'IRECI_6', 'VV_6', 'VH_6', 'B8_9', 'B4_9', 'B3_9', 'B2_9', 'B5_9', 'B6_9', 'B7_9', 'B11_9', 'B12_9', 'B11_1_9', 'B12_1_9', 'NDVI_9', 'EVI_9', 'IPVI_9', 'TNDVI_9', 'GNDVI_9', 'BI2_9', 'MTCI_9', 'REIP_9', 'IRECI_9', 'VV_9', 'VH_9', 'B8_12', 'B4_12', 'B3_12', 'B2_12', 'B5_12', 'B6_12', 'B7_12', 'B11_12', 'B12_12', 'B11_1_12', 'B12_1_12', 'NDVI_12', 'EVI_12', 'IPVI_12', 'TNDVI_12', 'GNDVI_12', 'BI2_12', 'MTCI_12', 'REIP_12', 'IRECI_12', 'VV_12', 'VH_12', 'Slope','Aspect']
print(bands)
samples = stack.sampleRegions(
    collection=training_fc,
    properties=['class'],
    scale=10,
    tileScale=4,
    geometries=True
)


['B8_3', 'B4_3', 'B3_3', 'B2_3', 'B5_3', 'B6_3', 'B7_3', 'B11_3', 'B12_3', 'B11_1_3', 'B12_1_3', 'NDVI_3', 'EVI_3', 'IPVI_3', 'TNDVI_3', 'GNDVI_3', 'BI2_3', 'MTCI_3', 'REIP_3', 'IRECI_3', 'VV_3', 'VH_3', 'B8_6', 'B4_6', 'B3_6', 'B2_6', 'B5_6', 'B6_6', 'B7_6', 'B11_6', 'B12_6', 'B11_1_6', 'B12_1_6', 'NDVI_6', 'EVI_6', 'IPVI_6', 'TNDVI_6', 'GNDVI_6', 'BI2_6', 'MTCI_6', 'REIP_6', 'IRECI_6', 'VV_6', 'VH_6', 'B8_9', 'B4_9', 'B3_9', 'B2_9', 'B5_9', 'B6_9', 'B7_9', 'B11_9', 'B12_9', 'B11_1_9', 'B12_1_9', 'NDVI_9', 'EVI_9', 'IPVI_9', 'TNDVI_9', 'GNDVI_9', 'BI2_9', 'MTCI_9', 'REIP_9', 'IRECI_9', 'VV_9', 'VH_9', 'B8_12', 'B4_12', 'B3_12', 'B2_12', 'B5_12', 'B6_12', 'B7_12', 'B11_12', 'B12_12', 'B11_1_12', 'B12_1_12', 'NDVI_12', 'EVI_12', 'IPVI_12', 'TNDVI_12', 'GNDVI_12', 'BI2_12', 'MTCI_12', 'REIP_12', 'IRECI_12', 'VV_12', 'VH_12', 'Slope', 'Aspect']


In [13]:
print(samples.first().getInfo())

{'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'Point', 'coordinates': [77.05600285155376, 31.943417766827093]}, 'id': '0_0', 'properties': {'Aspect': 276.0548095703125, 'B11_12': 828, 'B11_1_12': 828, 'B11_1_3': 953.5, 'B11_1_6': 1442, 'B11_1_9': 912.5, 'B11_3': 953.5, 'B11_6': 1442, 'B11_9': 912.5, 'B12_12': 472, 'B12_1_12': 472, 'B12_1_3': 502.5, 'B12_1_6': 809, 'B12_1_9': 469.5, 'B12_3': 502.5, 'B12_6': 809, 'B12_9': 469.5, 'B2_12': 60, 'B2_3': 16.5, 'B2_6': 1090, 'B2_9': 167.5, 'B3_12': 231, 'B3_3': 237.5, 'B3_6': 1098, 'B3_9': 316, 'B4_12': 232, 'B4_3': 177.5, 'B4_6': 869, 'B4_9': 177.5, 'B5_12': 463, 'B5_3': 527.5, 'B5_6': 1213, 'B5_9': 477.5, 'B6_12': 931, 'B6_3': 1234, 'B6_6': 2007, 'B6_9': 1153.5, 'B7_12': 1234, 'B7_3': 1485, 'B7_6': 2283, 'B7_9': 1364.5, 'B8_12': 1450, 'B8_3': 1573, 'B8_6': 2152, 'B8_9': 1335, 'B8_asm_12': 0.10416666666666669, 'B8_asm_3': 0.10416666666666669, 'B8_asm_6': 0.10416666666666669, 'B8_asm_9': 0.10416666666666669, 'B8_contrast_12': 366

In [14]:
classes = samples.aggregate_array('class').distinct()

train_list = []
test_list = []

for c in classes.getInfo():

    class_samples = samples.filter(ee.Filter.eq('class', c)).randomColumn('rand')
    
    size = class_samples.size().getInfo()

    # enforce 20% test for class 0,1,2
    if c in [0,1,2]:
        n_test = int(round(0.28 * size))
    else:
        n_test = int(round(0.28 * size))  # you can change this if you want

    sorted_fc = class_samples.sort('rand')

    test_part = sorted_fc.limit(n_test)
    train_part = sorted_fc.filter(ee.Filter.gte('rand',
                    test_part.aggregate_max('rand')))

    train_list.append(train_part)
    test_list.append(test_part)

train = ee.FeatureCollection(train_list).flatten()
test = ee.FeatureCollection(test_list).flatten()

In [15]:
print(train.aggregate_histogram('class').getInfo())
print(test.aggregate_histogram('class').getInfo())

{'0': 15, '1': 9, '2': 7, '3': 9, '4': 6, '5': 10, '6': 2}
{'0': 6, '1': 3, '2': 3, '3': 3, '4': 2, '5': 3, '6': 1}


In [16]:
classifier = ee.Classifier.smileRandomForest(77).train(
    features=train,
    classProperty='class',
    inputProperties=bands
)

In [17]:
classified = stack.classify(classifier)

In [18]:
test_classified = test.classify(classifier)

confusion_matrix = test_classified.errorMatrix('class','classification')

print("Confusion Matrix")
print(confusion_matrix.getInfo())

print("Overall Accuracy")
print(confusion_matrix.accuracy().getInfo())

Confusion Matrix
[[4, 1, 0, 1, 0, 0, 0], [0, 2, 1, 0, 0, 0, 0], [1, 0, 1, 0, 0, 1, 0], [1, 0, 0, 2, 0, 0, 0], [0, 0, 0, 0, 2, 0, 0], [0, 0, 0, 0, 0, 3, 0], [0, 0, 0, 0, 0, 0, 1]]
Overall Accuracy
0.7142857142857143


In [19]:
palette = [
'#006400',
'#1f78b4',
'#ff7f00',
'#6a3d9a',
'#e31a1c',
'#ffd92f',
'#17becf',
]

Map.addLayer(
    classified,
    {'min':0,'max':6,'palette':palette},
    "Tree Species Classification"
)

Map

Map(bottom=26857.0, center=[32.263910555201306, 77.3822021484375], controls=(WidgetControl(options=['position'…